In [1]:
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine('postgresql://localhost/customer_intelligence')

try:
    with engine.connect() as conn:
        print("Bağlantı başarılı.")
except Exception as e:
    print("Hata:", e)

df = pd.read_csv('telecom_ibm_feature_engineered.csv')
print(df.shape)

Bağlantı başarılı.
(7043, 43)


In [2]:
def get_contract_type(row):
    if row['Contract_Two year'] == 1:
        return 'Two year'
    elif row['Contract_One year'] == 1:
        return 'One year'
    else:
        return 'Month-to-month'

def get_internet_service(row):
    if row['InternetService_Fiber optic'] == 1:
        return 'Fiber optic'
    elif row['InternetService_No'] == 1:
        return 'No'
    else:
        return 'DSL'

def get_payment_method(row):
    if row['PaymentMethod_Electronic check'] == 1:
        return 'Electronic check'
    elif row['PaymentMethod_Mailed check'] == 1:
        return 'Mailed check'
    elif row['PaymentMethod_Credit card (automatic)'] == 1:
        return 'Credit card (automatic)'
    else:
        return 'Bank transfer (automatic)'

customers_for_db = pd.DataFrame({
    'customerid': df['customerID'],
    'name': df['Name'],
    'age': df['Age'],
    'gender': df['Gender_Binary'].map({1: 'Male', 0: 'Female'}),
    'contracttype': df.apply(get_contract_type, axis=1),
    'monthlycharge': df['MonthlyCharges'],
    'tenure': df['tenure'],
    'internetservice': df.apply(get_internet_service, axis=1),
    'paymentmethod': df.apply(get_payment_method, axis=1),
    'supportticketcount': df['SupportTicketCount']
})

print(customers_for_db.shape)
customers_for_db.head()

(7043, 10)


,customerid,name,age,gender,contracttype,monthlycharge,tenure,internetservice,paymentmethod,supportticketcount
0,7590-VHVEG,Şilan Bilge,56,Female,Month-to-month,29.85,1,DSL,Electronic check,5
1,5575-GNVDE,Taygan Gül,69,Male,One year,56.95,34,DSL,Mailed check,3
2,3668-QPYBK,Fami Sezer,46,Male,Month-to-month,53.85,2,DSL,Mailed check,6
3,7795-CFOCW,Denizalp Yıldırım,32,Male,One year,42.30,45,DSL,Bank transfer (automatic),2
4,9237-HQITU,Günsel Akar,60,Female,Month-to-month,70.70,2,Fiber optic,Electronic check,4


In [3]:
customers_for_db.to_sql('customers', engine, if_exists='append', index=False)
print(f"{len(customers_for_db)} müşteri kaydı yazıldı.")

7043 müşteri kaydı yazıldı.


In [4]:
check = pd.read_sql('SELECT COUNT(*) FROM customers', engine)
print(check)

sample = pd.read_sql('SELECT * FROM customers LIMIT 5', engine)
sample

   count
0   7043


,customerid,name,age,gender,contracttype,monthlycharge,tenure,internetservice,paymentmethod,supportticketcount
0,7590-VHVEG,Şilan Bilge,56,Female,Month-to-month,29.85,1,DSL,Electronic check,5
1,5575-GNVDE,Taygan Gül,69,Male,One year,56.95,34,DSL,Mailed check,3
2,3668-QPYBK,Fami Sezer,46,Male,Month-to-month,53.85,2,DSL,Mailed check,6
3,7795-CFOCW,Denizalp Yıldırım,32,Male,One year,42.30,45,DSL,Bank transfer (automatic),2
4,9237-HQITU,Günsel Akar,60,Female,Month-to-month,70.70,2,Fiber optic,Electronic check,4


In [5]:
churn_pred = pd.read_csv('churn_predictions.csv')
revenue_pred = pd.read_csv('revenue_predictions.csv')
segment_pred = pd.read_csv('customer_segments.csv')

print(churn_pred.shape, revenue_pred.shape, segment_pred.shape)
churn_pred.head()

(1409, 4) (1409, 3) (7043, 3)


,CustomerId,ActualChurn,PredictedChurn,ChurnProbability
0,1024-GUALD,1,1,0.70
1,0484-JPBRU,0,0,0.05
2,3620-EHIMZ,0,0,0.01
3,6910-HADCM,1,1,0.84
4,8587-XYZSF,0,0,0.05


In [6]:
merged = churn_pred.merge(revenue_pred, on='CustomerId', how='inner')
merged = merged.merge(segment_pred, left_on='CustomerId', right_on='customerID', how='inner')

print(merged.shape)
merged.head()

(1409, 9)


,CustomerId,ActualChurn,PredictedChurn,ChurnProbability,ActualRevenue,PredictedRevenue,customerID,Cluster,SegmentLabel
0,1024-GUALD,1,1,0.70,24.80,448.52,1024-GUALD,0,Yaslı Düşük Harcamalı Müşteri
1,0484-JPBRU,0,0,0.05,996.45,239.27,0484-JPBRU,3,Genç Düşük Harcamalı Müşteri
2,3620-EHIMZ,0,0,0.01,1031.70,547.96,3620-EHIMZ,0,Yaslı Düşük Harcamalı Müşteri
3,6910-HADCM,1,1,0.84,76.35,531.53,6910-HADCM,2,Yeni Yüksek Riskli Müşteri
4,8587-XYZSF,0,0,0.05,3260.10,2838.91,8587-XYZSF,1,Yüksek Değerli Sadık Müşteri


In [7]:
def get_risk_level(prob):
    if prob >= 0.7:
        return 'High'
    elif prob >= 0.4:
        return 'Medium'
    else:
        return 'Low'

predictions_for_db = pd.DataFrame({
    'customerid': merged['CustomerId'],
    'predictiondate': pd.Timestamp.now().strftime('%Y-%m-%d'),
    'churnprobability': merged['ChurnProbability'].round(4),
    'risklevel': merged['ChurnProbability'].apply(get_risk_level),
    'predictedrevenue': merged['PredictedRevenue'],
    'segmentlabel': merged['SegmentLabel']
})

print(predictions_for_db.shape)
predictions_for_db.head()

(1409, 6)


,customerid,predictiondate,churnprobability,risklevel,predictedrevenue,segmentlabel
0,1024-GUALD,2026-07-13,0.70,High,448.52,Yaslı Düşük Harcamalı Müşteri
1,0484-JPBRU,2026-07-13,0.05,Low,239.27,Genç Düşük Harcamalı Müşteri
2,3620-EHIMZ,2026-07-13,0.01,Low,547.96,Yaslı Düşük Harcamalı Müşteri
3,6910-HADCM,2026-07-13,0.84,High,531.53,Yeni Yüksek Riskli Müşteri
4,8587-XYZSF,2026-07-13,0.05,Low,2838.91,Yüksek Değerli Sadık Müşteri


In [8]:
predictions_for_db.to_sql('predictions', engine, if_exists='append', index=False)
print(f"{len(predictions_for_db)} tahmin kaydı yazıldı.")

1409 tahmin kaydı yazıldı.


In [9]:
check = pd.read_sql('SELECT COUNT(*) FROM predictions', engine)
print(check)

sample = pd.read_sql('SELECT * FROM predictions LIMIT 5', engine)
sample

   count
0   1409


,predictionid,customerid,predictiondate,churnprobability,risklevel,predictedrevenue,segmentlabel
0,1,1024-GUALD,2026-07-13,0.70,High,448.52,Yaslı Düşük Harcamalı Müşteri
1,2,0484-JPBRU,2026-07-13,0.05,Low,239.27,Genç Düşük Harcamalı Müşteri
2,3,3620-EHIMZ,2026-07-13,0.01,Low,547.96,Yaslı Düşük Harcamalı Müşteri
3,4,6910-HADCM,2026-07-13,0.84,High,531.53,Yeni Yüksek Riskli Müşteri
4,5,8587-XYZSF,2026-07-13,0.05,Low,2838.91,Yüksek Değerli Sadık Müşteri


In [11]:
churn_results = pd.read_csv('churn_model_performance.csv')
revenue_results = pd.read_csv('revenue_model_performance.csv')
segmentation_results = pd.read_csv('segmentation_model_performance.csv')

# Models tablosunun şemasına uygun hale getir
churn_for_db = pd.DataFrame({
    'modelname': churn_results['ModelName'],
    'algorithm': churn_results['Algorithm'],
    'accuracy': churn_results['Accuracy'],
    'precision_': churn_results['Precision'],
    'recall': churn_results['Recall'],
    'f1score': churn_results['F1Score'],
    'traindate': churn_results['TrainDate']
})

revenue_for_db = pd.DataFrame({
    'modelname': revenue_results['ModelName'],
    'algorithm': revenue_results['Algorithm'],
    'accuracy': None,
    'precision_': None,
    'recall': None,
    'f1score': revenue_results['Test_R2'],  # R² değerini f1score alanına koyuyoruz
    'traindate': revenue_results['TrainDate']
})

segmentation_for_db = pd.DataFrame({
    'modelname': ['Customer Segmentation (KMeans)'],
    'algorithm': segmentation_results['Algorithm'],
    'accuracy': None,
    'precision_': None,
    'recall': None,
    'f1score': segmentation_results['SilhouetteScore'],  # Silhouette'i f1score alanına koyuyoruz
    'traindate': pd.Timestamp.now().strftime('%Y-%m-%d')
})

all_models_for_db = pd.concat([churn_for_db, revenue_for_db, segmentation_for_db], ignore_index=True)
all_models_for_db

,modelname,algorithm,accuracy,precision_,recall,f1score,traindate
0,Logistic Regression,LogisticRegression,0.819021,0.680982,0.595174,0.635193,2026-07-10
1,Random Forest,RandomForestClassifier,0.789212,0.638686,0.469169,0.540958,2026-07-10
2,XGBoost,XGBClassifier,0.79489,0.638158,0.520107,0.573117,2026-07-10
3,Linear Regression,LinearRegression,None,None,None,0.757809,2026-07-10
4,Random Forest Regressor,RandomForestRegressor,None,None,None,0.780816,2026-07-10
5,Customer Segmentation (KMeans),KMeans,None,None,None,0.247300,2026-07-13


In [12]:
all_models_for_db.to_sql('models', engine, if_exists='append', index=False)
print(f"{len(all_models_for_db)} model kaydı yazıldı.")

6 model kaydı yazıldı.


In [13]:
check = pd.read_sql('SELECT * FROM models', engine)
check

,modelid,modelname,algorithm,accuracy,precision_,recall,f1score,traindate
0,1,Logistic Regression,LogisticRegression,0.819021,0.680982,0.595174,0.635193,2026-07-10
1,2,Random Forest,RandomForestClassifier,0.789212,0.638686,0.469169,0.540958,2026-07-10
2,3,XGBoost,XGBClassifier,0.794890,0.638158,0.520107,0.573117,2026-07-10
3,4,Linear Regression,LinearRegression,NaN,NaN,NaN,0.757809,2026-07-10
4,5,Random Forest Regressor,RandomForestRegressor,NaN,NaN,NaN,0.780816,2026-07-10
5,6,Customer Segmentation (KMeans),KMeans,NaN,NaN,NaN,0.247300,2026-07-13
